# CV gate teleportation
$\renewcommand{\coloneqq}{\mathrel{:=}}$
$\renewcommand{\Re}{\mathrm{Re}}$
$\renewcommand{\Im}{\mathrm{Im}}$

Quantum gate teleportation is the core of measurement based quantum computation as the implemented gate is depending on the measurement angles and the displacement correction (feedforward) needed on the output mode is calculated based on the measurement results.

In [69]:
import numpy as np
import piquasso as pq

dB = 77
SQUEEZING = np.log(10)/20*dB
HBAR = 2


#Displacement values for preparing coherent states
a = 5
b = 3
alpha = a + b*1j

c = 2
d = 7
beta = c + d*1j

One can implement $$\mathrm{\hat{G}}_{jk}(\mathbf{\theta})\coloneqq B^{\dagger}_{jk}
\hat{D}_j(\theta_1,\theta_3,m_1,m_3)\hat{V}_j(\theta_1, \theta_3)\hat{D}_k(\theta_2,\theta_4,m_2,m_4)\hat{V}_k(\theta_2, \theta_4)
B_{jk}$$
where
$$\hat{V}_i(\theta_h, \theta_l)=R_{i}(\theta^{l,h}_+)S_{i}(\ln\tan\theta^{l,h}_-, \pi)R_{i}(\theta^{l,h}_+)$$
$$\hat{D}_i(\theta_l,\theta_h,m_l,m_h) = D\left[ -\frac{e^{i\theta_h}m_l+e^{i\theta_l}m_h}{\sin(\theta_l - \theta_h)} \right]$$
where $m_i$ is a measurement result of $\hat{x}_{\theta_i}$.

Teleportation is only possible with some noise which decreases as we increase the the $r$ (`SQUEEZING`) parameter in the $\mathrm{EPR}$ state preparation.

Angle definitions are

$\theta_\pm^{l,h}\coloneqq\frac{\theta_l\pm\theta_h}{2}$
$\theta\coloneqq(\theta_1,\theta_2,\theta_3,\theta_4)$


Two single mode identical phaseshifters or squeezers commute with the 50:50 beamsplitter: $$B_{jk}R_j(\phi)R_k(\phi)=R_j(\phi)R_k(\phi)B_{jk}$$ $$B_{jk}S_j(r)S_k(r)=S_j(r)S_k(r)B_{jk}$$

Consequenty, the beamsplitters in $G$ should be undone manually, if two gates are not identical.


An other useful identity is $B_{nm}^\dagger=B_{mn}$, as a result 50:50 beamsplitters can be undone by changing the order of modes. 

Also $S(-r)=S(r, \pi)$ comes trivially from the definition of squeezing.

Moreover $B_{jk}D_j(\gamma)D_k(\delta)=B_{jk}D_{jk}(\gamma,\delta)=D_{jk}(U_{BS}(\gamma,\delta)^T)B_{jk}$ identity will become handy later in the feedforward. See `feedforward_j(x)` and `feedforward_k(x)`.

## Teleportation of certian gates

#### Phaseshifter
To teleport phaseshifters with a parameter $\phi$ the measurement angles $\theta$ should be choosen like
$$ \theta_+^{3,1}\coloneqq \theta_+^{4,2}\coloneqq \frac{\phi}{2} $$
$$ \theta_-^{3,1}\coloneqq \theta_-^{4,2}\coloneqq \frac{\pi}{4} $$
Then
$$ \theta_3 = \theta_4 = \theta_+^{3,1} + \theta_-^{3,1} = \frac{\phi}{2} +\frac{\pi}{4}$$
$$ \theta_1 = \theta_2 = \theta_+^{3,1} - \theta_-^{3,1} = \frac{\phi}{2} -\frac{\pi}{4}$$
$$\psi\coloneqq\theta_1+\frac{\pi}{2}=\theta_2+\frac{\pi}{2}=\theta_3=\theta_4$$
and $$\theta_+^{3,1}=\theta_+^{4,2}=\psi- \frac{\pi}{4}$$
the squeezers become identities ($\theta_-^{3,1}=\pi/4 \implies\tan\theta_-^{3,1}=1$), so $G(\theta)=R_{j}(\phi)R_{k}(\phi)$




In [70]:
# two mode identical phaseshifter teleport setup
def id_phaseshifter(phi):
    theta_plus = phi/2
    theta_minus = np.pi/4
    theta_3_4 = theta_plus+theta_minus
    theta_1_2 = theta_plus-theta_minus
    return [theta_1_2, theta_1_2, theta_3_4, theta_3_4]

Teleporting two different phaseshifters is quite similar except the sandwiching beamsplitters should be removed. 
$$ \psi_1\coloneqq\theta_3=\theta_1+\pi/2 
\\
\psi_2\coloneqq\theta_4=\theta_2+\pi/2 $$
 and $\phi_i\coloneqq2\psi_i- \frac{\pi} {2}, (i=1,2)$, the squeezers become identities (see above), $$\mathrm{\hat{G}}_{jk}(\theta)=B^{\dagger}_{jk}R_{j}(\phi_1)R_{k}(\phi_2)B_{jk}$$

In [71]:
# different valued phaseshifter teleport setup
def arb_phaseshiter(phi1, phi2):
    #Corrections 
    #Before: pq.Beamsplitter5050() modes in reverse order
    #After: pq.Beamsplitter5050()
    theta_plus_3_1 = phi1/2
    theta_plus_4_2 = phi2/2

    theta_minus = np.pi/4
    theta3 = theta_plus_3_1+theta_minus
    theta1 = theta_plus_3_1-theta_minus

    theta4 = theta_plus_4_2+theta_minus
    theta2 = theta_plus_4_2-theta_minus
    return [theta1, theta2, theta3, theta4]

#### Squeezing
To implement two identical squeezers with parameter $r$ one should choose $$-\theta_1\coloneqq-\theta_2\coloneqq\theta_3\coloneqq\theta_4 \coloneqq 
\arctan e^{r}=\phi$$
We get
$$\theta_-^{4,2}=\theta_-^{3,1} = \phi$$
for squeezing parameter and phaseshiter parameters trivially become
$$\theta_+^{4,2}=\theta_+^{3,1} = 0$$
Indeed $r=\ln\tan\phi$ so $\mathrm{G}$ simplifies to $$S_{j}(r,\pi)S_{k}(r,\pi)$$

In [72]:
# two mode identical squeezing teleport setup
def id_squeezing(r):
    #r = 3
    phi = np.arctan(np.exp(r))
    return [-phi, -phi, phi, phi]

In order to implement two different squeezers one should choose 
$$\arctan e^{r_1}=\phi_1\coloneqq\theta_3=-\theta_1$$
$$\arctan e^{r_2}=\phi_2\coloneqq\theta_4=-\theta_2$$
 and $r_i=\ln\tan\phi_i, (i=1,2)$, as G simplifies to $$B^{\dagger}_{jk}S_{j}(r_1,\pi)S_{k}(r_2,\pi)B_{jk}$$

In [73]:
# different valued squeezing
def arb_squeezing(r1, r2):
    #Corrections 
    #Before: pq.Beamsplitter5050() modes in reverse order
    #After: pq.Beamsplitter5050()
    phi1 = np.arctan(np.exp(r1))
    phi2 = np.arctan(np.exp(r2))
    return [-phi1, -phi2, phi1, phi2]



#### Beamsplitter

$\theta_- \coloneqq \psi$
$\theta_+ \coloneqq -\pi/2$ 

We choose

$$\theta_1 \coloneqq \frac{\theta_++\theta_-}{2} = -\frac{\pi}{4}+\frac{\psi}{2}
\\
\theta_2 \coloneqq \frac{\theta_+-\theta_-}{2} = -\frac{\pi}{4}-\frac{\psi}{2}
\\
\theta_3 \coloneqq \theta_1+\frac{\pi}{2}
\\
\theta_4 \coloneqq \theta_2+\frac{\pi}{2}$$

It gives
$$\theta_-^{3,1} = \frac{\theta_1+\pi/2-\theta_1}{2} = \frac{\pi}{4}
\\
\theta_-^{4,2} = \frac{\theta_2+\pi/2-\theta_2}{2} = \frac{\pi}{4}$$
Hence squeezings become trivial
$$\tan\frac{\pi}{4}=1 \implies S(-\ln\tan\frac{\pi}{4})=\mathbf{I}$$


For phaseshifter parameters we obtain 
$$\theta_+^{3,1} = \frac{\theta_1+\pi/2+\theta_1}{2} = \theta_1+\frac{\pi}{4} = \frac{\psi}{2}
\\
\theta_+^{4,2} = \frac{\theta_2+\pi/2+\theta_2}{2} = \theta_2+\frac{\pi}{4} = -\frac{\psi}{2}$$

The teleported gate:
$$\mathrm{\hat{G}}_{jk}(\theta) = B^{\dagger}_{jk}R_{j}(\psi)R_{k}(-\psi)B_{jk}= \\ = B_{jk}(\psi,-\frac{\pi}{2})$$
With before and after phase corrections we get get a fully parametrised beamsplitter

$$
\boxed{
\begin{aligned}
R_k(\frac{\pi}{2}+\phi)\mathrm{\hat{G}}_{jk}(\theta)R_k(-\frac{\pi}{2}-\phi) = \\ =
R_k(\frac{\pi}{2}+\phi)B_{jk}(\psi,-\frac{\pi}{2})R_k^\dagger(\frac{\pi}{2}+\phi) = \\ =
B_{jk}(\psi,\phi)
\end{aligned}
}
$$


Using $R_m(\alpha)B_{nm}(\theta, \phi)R_m^{\dagger}(\alpha)=B_{nm}(\theta, \phi+\alpha)$ idenity.

In [74]:
def real_beamsplitter(psi):
    # Corrections
    # Before pq.Phaseshifter(-np.pi/2-phi) on the 2nd mode
    # After pq.Phaseshifter(np.pi/2+phi) on the 2nd mode
    theta_minus_BS = psi
    theta_plus_BS = -np.pi/2
    theta1 = (theta_plus_BS+theta_minus_BS)/2
    theta2 = (theta_plus_BS-theta_minus_BS)/2
    return [theta1, theta2, theta1+np.pi/2, theta2+np.pi/2]




#### Clements Beamsplitter

$T_{j,k}(\gamma,\delta)$

$$
T_{j,k}(\gamma,\delta)
=
\begin{pmatrix}
e^{i\delta}\cos\gamma & -\sin\gamma\\
e^{i\delta}\sin\gamma & \cos\gamma
\end{pmatrix}=\\
=
R_k\!\left(\frac{\pi}{2}\right)
W_{j,k}(\pi,\gamma)
R_j\!\left(\delta+\frac{\pi}{2}\right)
=\\
=
\begin{pmatrix}
1 & 0\\
0 & i
\end{pmatrix}
\begin{pmatrix}
-i\cos\gamma & -\sin\gamma\\
-\sin\gamma & -i\cos\gamma
\end{pmatrix}
\begin{pmatrix}
i e^{i\delta} & 0\\
0 & 1
\end{pmatrix}
$$

$$
W_{jk}(\alpha, \beta) = B_{jk}^\dagger R_j(2\xi_+)R_k(2\xi_-)B_{jk}=\\
R_k(\frac{\pi}{2})B_{jk}(\beta)R_j(\frac{\pi}{2})R_j(\alpha)R_k(\alpha)
$$

$\xi_\pm\coloneqq\frac{1}{2}(\alpha\pm\beta-\frac{\pi}{2})$

$$\mathrm{\hat{G}}_{jk}(\theta)=B^{\dagger}_{jk}R_{k}(\phi_1)R_{k}(\phi_2)B_{jk}$$

$\phi_i\coloneqq2\psi_i- \frac{\pi} {2}, (i=1,2)$


$$ \psi_1\coloneqq\theta_3=\theta_1+\pi/2 
\\
\psi_2\coloneqq\theta_4=\theta_2+\pi/2 $$


$\alpha = \pi$
$\beta = \gamma$
$\mathrm{\hat{G}}_{jk}(\theta)=W_{jk}(\alpha,\beta)
\iff
\phi_{1}\coloneqq2\xi_+,
\phi_{2}\coloneqq2\xi_- 
$


$$
\\
2\psi_1-\frac{\pi}{2}\coloneqq2\xi_+=(\alpha+\beta-\frac{\pi}{2})
\\
2\psi_2-\frac{\pi}{2}\coloneqq2\xi_-=(\alpha-\beta-\frac{\pi}{2})
\\

\theta_3=\psi_1=\frac{\alpha+\beta}{2}=\frac{\pi}{2}+\frac{\beta}{2}
\\
\theta_4=\psi_2=\frac{\alpha-\beta}{2}=\frac{\pi}{2}-\frac{\beta}{2}
\\
\theta_1=\frac{\beta}{2}
\\
\theta_2=-\frac{\beta}{2}
$$


In [ ]:
def clements_bs(gamma):
    return [gamma/2, -gamma/2, np.pi+gamma/2, np.pi-gamma/2]


### Circuit

<div class="only-light wide-diagram">
  <img
    src="../_static/tutorials/two-mode-gate-teleport-light.svg"
    alt="Two-mode gate teleportation circuit"
    style="min-width: 150%; width: 150%; max-width: none; height: auto;"
  >
</div>

<div class="only-dark wide-diagram">
  <img
    src="../_static/tutorials/two-mode-gate-teleport-dark.svg"
    alt="Two-mode gate teleportation circuit"
    style="min-width: 150%; width: 150%; max-width: none; height: auto;"
  >
</div>

In [75]:
# Teleportation circuit
def gate_teleport_program(theta1, theta2, theta3, theta4):
    return pq.Program(instructions=gate_teleport_instructions(theta1, theta2, theta3, theta4))


def gate_teleport_instructions(theta1, theta2, theta3, theta4, mode_index = 0):
    
    def displacementFeedForward(x, theta_h, theta_l, i = 0):
        ml, mh = x[-4-2*i], x[-2-2*i]
        D_imag = (np.exp(1j*theta_h)*ml + np.exp(1j*theta_l)*mh)/(np.sin(theta_h - theta_l))
        D = D_imag/np.sqrt(HBAR)
        return D
    
    def displacement_j(x):
        return displacementFeedForward(x, theta1, theta3, 2)
    def displacement_k(x):
        return displacementFeedForward(x, theta2, theta4)
    

    def to_polar(a):
        return [np.abs(a), np.angle(a)]

    #commuted through beamsplitter
    def feedforward_j(x):
        D = (displacement_j(x) + displacement_k(x))/np.sqrt(2)
        return to_polar(D)

    
    def feedforward_k(x):
        D = (-displacement_j(x) + displacement_k(x))/np.sqrt(2)
        return to_polar(D)



    return [

        #0 - -> input1 M:theta3
        #1 - x squeezing M:theta1
        #2 - p squeezing -> output1
        #3 - -> input2 M:theta4
        #4 - x squeezing M:theta2
        #5 - p squeezing -> output2

        #EPR
        pq.Squeezing(SQUEEZING,0).on_modes(mode_index+1),
        pq.Squeezing(SQUEEZING,np.pi).on_modes(mode_index+2),
        pq.Beamsplitter5050().on_modes(mode_index+1, mode_index+2),


        #EPR
        pq.Squeezing(SQUEEZING, 0).on_modes(mode_index+4),
        pq.Squeezing(SQUEEZING, np.pi).on_modes(mode_index+5),
        pq.Beamsplitter5050().on_modes(mode_index+4, mode_index+5),


        #Foursplitter
        pq.Beamsplitter5050().on_modes(mode_index, mode_index+1),
        pq.Beamsplitter5050().on_modes(mode_index+3, mode_index+4),

        pq.Beamsplitter5050().on_modes(mode_index, mode_index+3),
        pq.Beamsplitter5050().on_modes(mode_index+1, mode_index+4),


        #Measurement
        pq.HomodyneMeasurement(phi=-theta3+np.pi/2, z=1e-5).on_modes(mode_index),
        pq.HomodyneMeasurement(phi=-theta1+np.pi/2, z=1e-5).on_modes(mode_index+1),
        pq.HomodyneMeasurement(phi=-theta4+np.pi/2, z=1e-5).on_modes(mode_index+3),
        pq.HomodyneMeasurement(phi=-theta2+np.pi/2, z=1e-5).on_modes(mode_index+4),
        
        #Feedforward
        pq.Displacement(lambda x : feedforward_j(x)[0], lambda x : feedforward_j(x)[1]).on_modes(mode_index+2),
        pq.Displacement(lambda x : feedforward_k(x)[0], lambda x : feedforward_k(x)[1]).on_modes(mode_index+5),

        #Phase corrections
        pq.Phaseshifter(-np.pi/2).on_modes(mode_index+2),
        pq.Phaseshifter(-np.pi/2).on_modes(mode_index+5)
    ]


Parameters are set here. The functions `id_phaseshifter(phi)`, `arb_phaseshiter(phi1, phi2)`, `id_squeezing(r)`, `arb_squeezing(r1, r2)`,
`real_beamsplitter(psi)` are used set `thetas` ($\theta$) accordingly. 

In [76]:
gamma = np.pi/11
delta = 3*np.pi/7
#phi = 3*np.pi/7

thetas = arb_phaseshiter(gamma, delta)

In [77]:
def prep():

    r1 = np.abs(alpha)
    phi1 = np.angle(alpha)
    r2 = np.abs(beta)
    phi2 = np.angle(beta)
    return pq.Program(
        instructions=[pq.Vacuum().on_modes(0, 1), 
                    pq.Displacement(r1, phi1).on_modes(0), 
                    pq.Displacement(r2, phi2).on_modes(1)])

with pq.Program() as TwoModeTest:
    pq.Q(0, 1) | prep()
    # general case
    #pq.Q(0, 1) | teleported_gates_program(*thetas)

    # two mode identical phaseshifter teleport
    #pq.Q(0) | pq.Phaseshifter(gamma)
    #pq.Q(1) | pq.Phaseshifter(gamma)

    # two mode identical squeezing teleport
    #pq.Q(0) | pq.Squeezing(r, np.pi)
    #pq.Q(1) | pq.Squeezing(r, np.pi)

    # different valued phaseshifter
    pq.Q(0, 1) | pq.Beamsplitter5050()
    pq.Q(0) | pq.Phaseshifter(gamma)
    pq.Q(1) | pq.Phaseshifter(delta)
    pq.Q(1, 0) | pq.Beamsplitter5050()


    # different valued squeezing
    #pq.Q(0, 1) | pq.Beamsplitter5050()
    #pq.Q(0) | pq.Squeezing(r1, np.pi)
    #pq.Q(1) | pq.Squeezing(r2, np.pi)
    #pq.Q(1, 0) | pq.Beamsplitter5050()
    


    #beamsplitter
    #pq.Q(0, 1) | pq.Beamsplitter(gamma, phi)

    

with pq.Program() as TwoDimClusterGateTeleport:
    pq.Q(0, 1, 2, 3, 4, 5) | pq.Vacuum()
    pq.Q(0, 3) | prep()

    pq.Q(3) | pq.Phaseshifter(-np.pi/2-phi)

    pq.Q(0, 1, 2, 3, 4, 5) | gate_teleport_program(*thetas)

    pq.Q(5) | pq.Phaseshifter(phi+np.pi/2)


### Simulation

In [78]:
# General case
def G(theta1, theta2, theta3, theta4):

    inst = [pq.Beamsplitter5050().on_modes(0, 1)]
    inst.extend(V(theta1, theta3, 0))
    inst.extend(V(theta2, theta4, 1))
    inst.extend([pq.Beamsplitter5050().on_modes(1, 0)])
    return pq.Program(instructions=inst)



def V(theta_l, theta_h, mode_index):

    plus_minus = lambda x, y: ((x+y)/2, (x-y)/2)
    theta_plus, theta_minus = plus_minus(theta_h, theta_l)
    rr1 = np.log(np.tan(theta_minus))
    return [
        pq.Phaseshifter(theta_plus).on_modes(mode_index),
        pq.Squeezing(rr1, np.pi).on_modes(mode_index),
        pq.Phaseshifter(theta_plus).on_modes(mode_index)
    ]


In [79]:
config = pq.Config(hbar=HBAR)


def make_simulator(d, seed=1234):
    return pq.GaussianSimulator(
        d=d,
        config=pq.Config(
            hbar=HBAR,
            seed_sequence=seed,
        ),
    )


simulator = make_simulator(2)

result = simulator.execute(TwoModeTest, shots=1)
target_state = result.state


teleport_simulator = make_simulator(6)
teleported_result = teleport_simulator.execute(TwoDimClusterGateTeleport,  shots=1)
teleported_state = teleported_result.state



#### Results and errors

Errors $\to 0$ as `SQUEEZING` $\to \infty$

In [80]:
print("Target mean:")
print(target_state.xxpp_mean_vector)

print("Target covariance:")
print(target_state.xxpp_covariance_matrix)



print("Teleported mean:")
print(teleported_state.xxpp_mean_vector)

print("Mean error:")
mean_err = (teleported_state.xxpp_mean_vector - target_state.xxpp_mean_vector)
print(mean_err)
print("Mean error norm: ", np.linalg.norm(mean_err))


print("Cov error:")

cov_err = (teleported_state.xxpp_covariance_matrix - target_state.xxpp_covariance_matrix)
print(cov_err)

print("Covarience error norm: ", np.linalg.norm(cov_err))



Target mean:
[ -4.18622344 -12.19704173   6.0569305   12.04247895]
Target covariance:
[[2. 0. 0. 0.]
 [0. 2. 0. 0.]
 [0. 0. 2. 0.]
 [0. 0. 0. 2.]]
Teleported mean:
[ 7.46820428 -1.09197857 14.91554641  8.28049728]
Mean error:
[11.65442772 11.10506316  8.85861591 -3.76198167]
Mean error norm:  18.7556843375686
Cov error:
[[ 2.23517418e-08  4.14477538e-10  3.25962901e-09  1.81594475e-09]
 [ 4.14477538e-10  2.26069683e-08  4.61692503e-09 -2.14140775e-09]
 [ 1.86264515e-09  4.61692503e-09  5.06406650e-09  1.39410475e-09]
 [ 1.81594475e-09 -7.44423891e-10  1.39410475e-09  4.80883999e-09]]
Covarience error norm:  3.36472503181612e-08


## One dimentional gate teleportation proof

<div class="only-light wide-diagram">
  <img
    src="../_static/tutorials/one-mode-gate-teleport-light.svg"
    alt="Two-mode gate teleportation circuit"
    style="min-width: 150%; width: 150%; max-width: none; height: auto;"
  >
</div>

<div class="only-dark wide-diagram">
  <img
    src="../_static/tutorials/one-mode-gate-teleport-dark.svg"
    alt="Two-mode gate teleportation circuit"
    style="min-width: 150%; width: 150%; max-width: none; height: auto;"
  >
</div>

#### Squeezing layer
$B$ and $C$ are squeezed in $\hat{p}$ and $\hat{x}$ respectively. 
$$
\hat x_B^{(1)}=e^{-r}\hat x_B^{(0)}\\
\hat x_C^{(1)}=e^{r}\hat x_C^{(0)}\\
\hat p_B^{(1)}=e^{r}\hat p_B^{(0)}\\
\hat p_C^{(1)}=e^{-r}\hat p_C^{(0)}
$$

#### Beamsplitter ($\mathrm{EPR}$ state prep)
    
Equal beamsplitter action on quadrature operators is described by
$U = 
    \frac{1}{\sqrt{2}}
    \begin{pmatrix}
    1 & -1 \\
    1 & 1 
    \end{pmatrix}$


Beamsplitter acting on $B$ and $C$ modes and substituting $\mathrm{EPR}$ coorrelations. 
Let us consider action the position operators

$$
\hat x_B^{(2)}=
\frac{1}{\sqrt2}
\left(
    \hat x_B^{(1)}
-
\hat x_C^{(1)}
\right)=
\frac{1}{\sqrt2}
\left(
e^{-r}\hat x_B^{(0)}
-
e^{r}\hat x_C^{(0)}
\right)\\
\hat x_C^{(2)}=
\frac{1}{\sqrt2}
\left(
\hat x_B^{(1)}
+
\hat x_C^{(1)}
\right)=
\frac{1}{\sqrt2}
\left(
e^{-r}\hat x_B^{(0)}
+
e^{r}\hat x_C^{(0)}
\right)\\
$$

Let us consider action the position operators

$$
\hat p_B^{(2)}=
\frac{1}{\sqrt2}
\left(
    \hat p_B^{(1)}
-
\hat p_C^{(1)}
\right)=
\frac{1}{\sqrt2}
\left(
e^{r}\hat p_B^{(0)}
-
e^{-r}\hat p_C^{(0)}
\right)\\
\hat p_C^{(2)}=
\frac{1}{\sqrt2}
\left(
\hat p_B^{(1)}
+
\hat p_C^{(1)}
\right)=
\frac{1}{\sqrt2}
\left(
    e^{r}\hat p_B^{(0)}
+
e^{-r}\hat p_C^{(0)}
\right)
$$
This gate leaves upper ($A$) mode unchanged 
$$
    \hat{x}_{in}\coloneqq
    \hat x_A^{(2)} = \hat x_A^{(1)} = \hat x_A^{(0)}\\
    \hat{p}_{in}\coloneqq
    \hat p_A^{(2)} = \hat p_A^{(1)} = \hat p_A^{(0)}\\
$$
Considering infinite squeezing $ r\to\infty $ 
$$
    \hat{x}_B^{(2)}+\hat{x}_C^{(2)}=\\
    =\frac{1}{\sqrt2}
\left(
e^{-r}\hat x_B^{(0)}
-
e^{r}\hat x_C^{(0)}
+
e^{-r}\hat x_B^{(0)}
+
e^{r}\hat x_C^{(0)}
\right)=\\
    \sqrt2e^{-r}\hat{x}_B^{(0)} \to 0
$$

$$
    \hat p_B^{(2)}-\hat p_C^{(2)}=\\
=\frac{1}{\sqrt2}
\left(
e^{r}\hat p_B^{(0)}
-
e^{-r}\hat p_C^{(0)}
-
e^{r}\hat p_B^{(0)}
-
e^{-r}\hat p_C^{(0)}
\right)=\\
-\sqrt2
e^{-r}\hat p_C^{(0)} \to 0
$$
We get the $\mathrm{EPR}$ (anti)correlations $x_B^{(2)}\approx-\hat x_C^{(2)}$ and $p_B^{(2)}\approx\hat p_C^{(2)}$.


#### Beamsplitter (mixing with the input)
Beamsplitter acting on $A$ and $B$ modes and substituting $\mathrm{EPR}$ coorrelations. 
Let us consider action the position operators
$$
    \hat{x}_A^{(3)} = \frac{1}{\sqrt2}(\hat{x}_A^{(2)}-\hat{x}_B^{(2)}) = \frac{1}{\sqrt2}(\hat{x}_{in}^{(0)}+\hat{x}_C^{(2)}) \\
    \hat{x}_B^{(3)} = \frac{1}{\sqrt2}(\hat{x}_A^{(2)}+\hat{x}_B^{(2)}) = \frac{1}{\sqrt2}(\hat{x}_{in}^{(0)}-\hat{x}_C^{(2)}) \\
$$
Same on the momentum operators
$$
    \hat{p}_A^{(3)} = \frac{1}{\sqrt2}(\hat{p}_A^{(2)}-\hat{p}_B^{(2)}) = \frac{1}{\sqrt2}(\hat{p}_{in}^{(0)}-\hat{p}_C^{(2)}) \\
    \hat{p}_B^{(3)} = \frac{1}{\sqrt2}(\hat{p}_A^{(2)}+\hat{p}_B^{(2)}) = \frac{1}{\sqrt2}(\hat{p}_{in}^{(0)}+\hat{p}_C^{(2)}) \\
$$
Leaves the $C$ mode unchanged
$$
    \hat{x}_C^{(3)} = \hat{x}_C^{(2)}\\
    \hat{p}_C^{(3)} = \hat{p}_C^{(2)}
$$
We get this from correlations
$$
    \hat{x}_A^{(3)}+\hat{x}_B^{(3)}=\sqrt2\hat{x}_A^{(0)}
$$
$$
    \hat{p}_A^{(3)}+\hat{p}_B^{(3)}=\sqrt2\hat{p}_A^{(0)}
$$

#### Measurement
Measurement is done in a rotated momentum operator $\hat{p}_\theta=\hat{p}\cos(\theta)+\hat{x}\sin(\theta)$.
Let us introduce 
$\hat{x}_{out}\coloneqq\hat{x}_C^{(2)}$, $\hat{p}_{out}\coloneqq\hat{p}_C^{(2)}$

So the measurement outcomes are
$$
    m_3 = \hat{p}_A^{(3)}\cos{\theta_3}+\hat{x}_A^{(3)}\sin{\theta_3} =\\
    \frac{1}{\sqrt2}
    \left[
    (\hat{p}_{in}-\hat{p}_{out})\cos{\theta_3}+
    (\hat{x}_{in}+\hat{x}_{out})\sin{\theta_3}
    \right]\\ 
    m_1 = \hat{p}_B^{(3)}\cos{\theta_1}+\hat{x}_B^{(3)}\sin{\theta_1} =\\
    \frac{1}{\sqrt2}
    \left[
    (\hat{p}_{in}+\hat{p}_{out})\cos{\theta_1}+
    (\hat{x}_{in}-\hat{x}_{out})\sin{\theta_1}
    \right]
$$

More elegantly
$$
    \sqrt2\begin{pmatrix}
        m_3\\
        m_1
    \end{pmatrix}
    =
    \begin{pmatrix}
        \sin{\theta_3} & -\cos{\theta_3}\\
        -\sin{\theta_1} & \cos{\theta_1}
    \end{pmatrix}
    \begin{pmatrix}
        \hat{x}_{out}\\
        \hat{p}_{out}
    \end{pmatrix}
    +
    \begin{pmatrix}
        \sin{\theta_3} & \cos{\theta_3} \\
        \sin{\theta_1} & \cos{\theta_1} 
    \end{pmatrix}
    \begin{pmatrix}
        \hat{x}_{in}\\
        \hat{p}_{in}
    \end{pmatrix}\\
$$
Express $\hat{x}_{out}$, $\hat{p}_{out}$ in terms of $\hat{x}_{in}$, $\hat{p}_{in}$ and $m_3$, $m_1$.
$$
    \begin{pmatrix}
        \hat{x}_{out}\\
        \hat{p}_{out}
    \end{pmatrix}=
    \frac{1}{\sin{(\theta_1-\theta_3)}}
    \begin{pmatrix}
        \cos{\theta_1} & \cos{\theta_3} \\
        \sin{\theta_1} & \sin{\theta_3} 
    \end{pmatrix}
    \begin{pmatrix}
        \sin{\theta_3} & \cos{\theta_3}\\
        \sin{\theta_1} & \cos{\theta_1}
    \end{pmatrix}
    \begin{pmatrix}
        \hat{x}_{in}\\
        \hat{p}_{in}
    \end{pmatrix}
    -\\
    -
    \frac{\sqrt2}{\sin{(\theta_1-\theta_3)}}
    \begin{pmatrix}
        \cos{\theta_1} & \cos{\theta_3} \\
        \sin{\theta_1} & \sin{\theta_3} 
    \end{pmatrix}
    \begin{pmatrix}
        m_3\\
        m_1
    \end{pmatrix}=\\=
         \frac{1}{\sin{(\theta_1-\theta_3)}}
    \begin{pmatrix}
        \sin{(\theta_3+\theta_1)} & 2\cos{\theta_3}\cos{\theta_1} \\
         2\sin{\theta_3}\sin{\theta_1} & \sin{(\theta_3+\theta_1)}
    \end{pmatrix}
    -
    \frac{\sqrt2}{\sin{(\theta_1-\theta_3)}}
    \begin{pmatrix}
        \Re Z \\
        \Im Z 
    \end{pmatrix}
$$

Gate teleportation can be proved by checking this
$$
    \frac{1}{\sin(\theta_1-\theta_3)}
    \begin{pmatrix}
    \sin(\theta_1+\theta_3)
    &
    2\cos\theta_1\cos\theta_3
    \\
    2\sin\theta_1\sin\theta_3
    &
    \sin(\theta_1+\theta_3)
    \end{pmatrix}
    =
    \mathrm{R}\left(\theta_+-\frac{\pi}{2}\right)
    \mathrm{S}\left(\ln\tan\theta_-\right)
    \mathrm{R}(\theta_+).
$$

The displacement feedforward can be seen 
$$-\frac{\sqrt2}{\sin{(\theta_1-\theta_3)}}
    \begin{pmatrix}
        \Re Z \\
        \Im Z 
    \end{pmatrix}$$
 where $Z = \exp(i\theta_3)m_1+\exp(i\theta_1)m_3$

példa
1 módusú
teleportációt elmagyarázni, graphical calculus